# 04 · Clusters, factores latentes y gradientes
Este notebook responde: **¿qué regímenes comerciales subyacentes existen?** y **¿qué variables mueven la predicción del modelo?**

Los clusters son descriptivos. Los gradientes son sensibilidad del modelo, no elasticidad causal.

In [ ]:
from pathlib import Path
import sys, pandas as pd, numpy as np, matplotlib.pyplot as plt
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src'/'replica_cygnus').exists())
sys.path.insert(0, str(ROOT/'src'))
from replica_cygnus.economic_intelligence import install_feature_mart, load_monthly_panel, build_supervised_panel, cluster_project_regimes, finite_difference_gradients
from replica_cygnus.economic_intelligence.features import select_model_matrix
from replica_cygnus.economic_intelligence.models import fit_champion_model
install_feature_mart(); panel = load_monthly_panel(); sup = build_supervised_panel(panel)


In [ ]:
regimes, cluster_model, pca_model = cluster_project_regimes(panel, n_clusters=4)
regimes.groupby('regime_cluster').size().rename('n_project_months').to_frame()

In [ ]:
fig, ax = plt.subplots(figsize=(9,6))
for c, g in regimes.groupby('regime_cluster'):
    ax.scatter(g['latent_factor_1'], g['latent_factor_2'], s=28, alpha=.65, label=f'Cluster {c}')
ax.set_title('Regímenes latentes proyecto-mes (PCA + KMeans)'); ax.set_xlabel('Factor latente 1'); ax.set_ylabel('Factor latente 2'); ax.legend(); ax.grid(alpha=.15); plt.show()

## Perfil económico de cada cluster
Cruzar los labels con stock, absorción, caídas y edad comercial convierte un cluster matemático en un régimen interpretable: lanzamiento, aceleración, madurez, stock lento, etc.

In [ ]:
profile = panel.reset_index().merge(regimes[['periodo_mes','codigo_proyecto','regime_cluster']], on=['periodo_mes','codigo_proyecto'], how='inner')
profile.groupby('regime_cluster').agg(stock=('stock_inicio_observado','mean'), mov_neto=('movimiento_neto_mes','mean'), absorcion=('absorcion_neta_mes','mean'), caidas=('caidas_mes','mean'), edad=('edad_comercial_meses','mean')).round(3)

## Gradiente predictivo
Entrenamos un champion rápido para el movimiento neto del próximo mes. Luego perturbamos cada feature `+ε/-ε` y aproximamos `∂ŷ/∂x` por diferencias finitas.

In [ ]:
target = 'target_mov_neto_next_1m'
X, y, feature_cols = select_model_matrix(sup, target, include_macro=True, include_reference=False)
model = fit_champion_model(X, y, model_name='hist_gradient_boosting')
grad = finite_difference_gradients(model, X.tail(min(100, len(X))), features=feature_cols)
grad.head(15)

In [ ]:
top = grad.head(10).sort_values('gradient_abs_mean')
fig, ax = plt.subplots(figsize=(9,5))
ax.barh(top['feature'], top['gradient_abs_mean'])
ax.set_title('Sensibilidad media absoluta del modelo'); ax.set_xlabel('|∂ predicción / ∂ feature|'); plt.show()

### Fiscalización
1. Un cluster no es un segmento causal.
2. Un gradiente de ML no es elasticidad de precio.
3. Si una variable actual de referencia domina un modelo histórico, revisar leakage.
4. Para decisiones de precio, exigir historial de precios o experimento identificado.